In [ ]:
!pip install -q groq pandas
import os
import zipfile
import pandas as pd
from groq import Groq

# ==========================
# Groq API
# ==========================
GROQ_API_KEY = ""

client = Groq(api_key=GROQ_API_KEY)

# ==========================
# Load Dataset
# ==========================
ZIP_FILE = "/content/NaukriData_Data Science.csv.zip"

with zipfile.ZipFile(ZIP_FILE, 'r') as z:
    csv_file = z.namelist()[0]
    with z.open(csv_file) as f:
        df = pd.read_csv(f)

print("Dataset Loaded Successfully")
print(df.head())

# ==========================
# Search Function
# ==========================
def search_jobs(question):

    question = question.lower()

    mask = df.astype(str).apply(
        lambda col: col.str.lower().str.contains(question, na=False)
    )

    results = df[mask.any(axis=1)]

    if len(results) == 0:
        return None

    return results.head(15)

# ==========================
# Chatbot
# ==========================
def chatbot(question):

    results = search_jobs(question)

    if results is None:
        return "No matching job found."

    context = results.to_string(index=False)

    prompt = f"""
You are a Job Recommendation Assistant.

Use ONLY the job records below.

JOB DATA

{context}

QUESTION

{question}

Answer clearly.
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0,
        max_tokens=400
    )

    return response.choices[0].message.content

# ==========================
# Chat Loop
# ==========================
print("\nJob Chatbot Ready")
print("Type exit to stop")

while True:

    q = input("\nYou : ")

    if q.lower() == "exit":
        break

    ans = chatbot(q)

    print("\nBot :", ans)

Dataset Loaded Successfully
                                          Job_Titles  \
0                      Senior Manager - Data Science   
1      Advance Analytical and Data Sciences -Manager   
2  Manager - Digital Product Analytics [Data Scie...   
3                               Data Science Manager   
4                               Data Science Manager   

                       Company_Names Experience_Required Package_Details  \
0                   AMERICAN EXPRESS             4-8 Yrs   Not disclosed   
1                  G R Infraprojects            6-10 Yrs   Not disclosed   
2                               Resy             4-8 Yrs   Not disclosed   
3          Foreign IT Consulting MNC             1-3 Yrs   Not disclosed   
4  Fortune India 500 Company in FMCG             4-9 Yrs   Not disclosed   

                                           Locations  \
0                                   Gurgaon/Gurugram   
1                         Gurgaon/ Gurugram, Haryana   
2         